 # 🧾 Day 3 – Conversational Memory Bot

We move from stateless RAG → to stateful conversational systems.

> Before we build a Conversational memory bot, Lets test whether the LLM is capable of remembering the previous conversations.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0 # pushing it to not be creative
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

chain = prompt | llm

response1 = chain.invoke({"input": "My name is Koushik."})
print(response1.content)

Nice to meet you, Koushik. How can I assist you today?


In [3]:
response2 = chain.invoke({"input": "What is my name?"})
print(response2.content)

I don't have any information about your name. I'm a new conversation each time you interact with me, so I don't retain any personal data or context from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!


### Observations:

The LLM cannot remember any response from the previous conversation and is strictly limited to context provided.

***LLM = Intelligence component***

***LLM ≠ Application***

Inorder to achieve this, we need to store all the past conversations.

For this, we need to use LangGraph

---
## What LangGraph Is

> LangGraph is a stateful graph engine for LLM workflows — unlike traditional stateless LLM calls, it lets you define a graph of nodes that share and update state, enabling conversational memory and multi-turn behavior.

Key idea:

State = your bot’s memory that persists across invocations
Nodes = functions that update state
Edges = define the flow (order nodes execute in)

In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

# Define state schema using MessagesState
class ChatState(MessagesState):
    ...

# 1️⃣ Initialize your Groq LLM
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# 2️⃣ Define node to append user messages
def user_node(state: ChatState):
    # Node returns nothing — state already contains new messages passed in
    return {}

# 3️⃣ Define node to let the LLM reply
def llm_response_node(state: ChatState):
    # LangGraph holds state["messages"] as a growing list of messages
    # We call the LLM with the entire message history so far.
    reply = llm.invoke(state["messages"])
    # Return AI message so MessagesState automatically appends it
    return {"messages": [AIMessage(content=reply.content)]}

# 4️⃣ Build the graph
builder = StateGraph(ChatState)

builder.add_node("user_node", user_node)
builder.add_node("llm_response_node", llm_response_node)

builder.add_edge(START, "user_node")
builder.add_edge("user_node", "llm_response_node")
builder.add_edge("llm_response_node", END)

# Short-term memory via checkpoint
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# 5️⃣ Chat with memory
session_id = "chat_session_1"
config = {"configurable": {"thread_id": session_id}}

# First user turn
graph.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Alice!")]},
    config,
)

# Second user turn
result = graph.invoke(
    {"messages": [HumanMessage(content="What’s my name?")]},
    config,
)

# Print the agent's reply
print(result["messages"][-1].content)

Your name is Alice. You told me that when we first started talking.


```
START
  │
  ▼
Load last checkpoint for thread_id
  │
  ▼
Add new HumanMessage to state["messages"]
  │
  ▼
NODE: user_node
  │  (Optional placeholder — state already updated)
  ▼
NODE: llm_response_node
  │  (call Groq LLM with full state["messages"])
  │
  ▼
LLM generates response
  │
  ▼
Append AIMessage to state["messages"]
  │
  ▼
Save updated state as new checkpoint for thread_id
  │
  ▼
Return current state["messages"]
END
```

In [5]:
test_human_messages = [
    HumanMessage(content="Hello!"),
    HumanMessage(content="Hi, can you help me?"),
    HumanMessage(content="What's my name?"),
    HumanMessage(content="Tell me a fun fact about space."),
    HumanMessage(content="How do I cook pasta?"),
    HumanMessage(content="Remind me of my favorite color."),
    HumanMessage(content="What did I say earlier in this conversation?"),
    HumanMessage(content="Can you summarize our chat so far?"),
    HumanMessage(content="I love painting and reading books."),
    HumanMessage(content="My birthday is June 10th."),
]

In [6]:
for msg in test_human_messages:
    result = graph.invoke(
        {"messages": msg.content},
        config,
    )



In [7]:
for i in range(len(result["messages"])):
    msg = result["messages"][i]
    if isinstance(msg, HumanMessage):
        role = "user"
    elif isinstance(msg, AIMessage):
        role = "assistant"
    else:
        role = "unknown"
    print(f"{role}: {msg.content}")
    if i%2 != 0: print("\n")

user: Hi, my name is Alice!
assistant: Hello Alice! It's nice to meet you. Is there something I can help you with or would you like to chat?


user: What’s my name?
assistant: Your name is Alice. You told me that when we first started talking.


user: Hello!
assistant: Hello again, Alice! How's your day going so far?


user: Hi, can you help me?
assistant: I'd be happy to help you, Alice. What do you need help with? Do you have a question, a problem, or just need someone to talk to?


user: What's my name?
assistant: Your name is Alice. We've been chatting for a bit, and you told me your name at the beginning of our conversation.


user: Tell me a fun fact about space.
assistant: Alice, here's a fun fact about space: Did you know that there is a giant storm on Jupiter that has been raging for at least 150 years? It's called the Great Red Spot, and it's so large that three Earths could fit inside it. Isn't that amazing?


user: How do I cook pasta?
assistant: Cooking pasta is a straight